# 01 — Data Collection & Wrangling (ETL)

**Objective:** Extract the raw policyholder CSV (`Dataset/raw_policyholders.csv`), inspect real-world data quality anomalies, clean and normalize the dataset, engineer `annual_value_rm`, and load the result into a relational SQLite database (`data/insurai.db`).

In [ ]:
import sys
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.etl_pipeline import (
    DEFAULT_RAW_CSV,
    DEFAULT_DB_PATH,
    extract_raw_data,
    transform_policyholders,
    load_to_sqlite,
)

## 1. Extract & Inspect Raw Data Anomalies

In [ ]:
raw_df = extract_raw_data(DEFAULT_RAW_CSV)
print(f"Raw shape: {raw_df.shape}")
display(raw_df.head())

print("\nMissing values per column:")
print(raw_df.isna().sum())

print("\nRaw distinct region values:")
print(sorted(raw_df["region"].dropna().unique().tolist()))

## 2. Transform (Normalize Strings, Impute Missing Values, Feature Engineering)

In [ ]:
clean_df = transform_policyholders(raw_df)
print(f"Cleaned shape: {clean_df.shape}")
print("\nCleaned distinct region values:")
print(sorted(clean_df["region"].unique().tolist()))
print("\nMissing values after cleaning:")
print(clean_df.isna().sum())
display(clean_df.head())

## 3. Load into SQLite (`data/insurai.db`) & Verify

In [ ]:
stats = load_to_sqlite(clean_df, DEFAULT_DB_PATH)
print(f"Loaded into {DEFAULT_DB_PATH}: {stats}")

with sqlite3.connect(DEFAULT_DB_PATH) as conn:
    preview = pd.read_sql_query("SELECT * FROM policyholders LIMIT 5;", conn)
display(preview)

## Key Insights

- Raw `region` values contained leading/trailing whitespace and mixed casing (e.g., `' Penang '`, `'selangor'`, `'KUALA LUMPUR'`), which were standardized into 5 clean Malaysian regions.
- Missing `late_payments` entries were imputed with `0` and cast to integers; blank `churn_reason` entries (for retained customers where `churned == 0`) were set to `'Not Applicable'` for zero-`NULL` SQL storage.
- `annual_value_rm` (`monthly_premium_rm * 12`) was engineered to quantify annual recurring premium at risk in downstream SQL EDA and the Streamlit dashboard.
- **Modeling Note:** `churn_reason` is retained in SQLite for descriptive EDA only and must be dropped alongside `policy_id` before ML training to prevent target leakage.